# RFM + K-Means Tourism Guest Segmentation – Practice Skeleton

**Short name (GitHub):** `TourRFM`

**Lab source:** CustSeg / BankRFM adapted to hotel and tour bookings — clean stays, build Recency / Frequency / Monetary, log+scale, K-Means, PCA view.

Work this notebook first. Peek at `TourRFM_Solution.ipynb` only when stuck. `TourRFM.py` is the reusable helper (sklearn if present, NumPy otherwise).

**Files**
- `data/tour_bookings.csv` — 14,354 line items (synthetic hotel/tour extract, Dec 2010–Dec 2011)
- `tourrfm_flowchart.png` — desired outcome
- `TourRFM_Cheatsheet.docx`, `TourRFM_Project_Memo.docx`, `TourRFM_Strategy_Guide.docx`

**Not a yield / pricing decision, not a visa decision, not a rate-parity engine.** Clustering groups similar RFM rows. A human still names the bins and owns the treatment.


## Inline cheat-sheet (keep this cell visible)

See also **`TourRFM_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Line item vs guest | row in `tour_bookings.csv` = one booking; RFM row = one `GuestID` |
| TotalSum | \(Q \times\) UnitRate after `Quantity > 0` (and usually `UnitRate > 0`) |
| Snapshot | \(\text{snapshot} = \max(\text{StayDate}) + 1\text{ day}\) |
| Recency | \((\text{snapshot} - \max_i t_i).\text{days}\) — smaller is warmer |
| Frequency | `nunique(BookingNo)`, **not** product-line `count` |
| Monetary | \(\sum Q\cdot A\) over the window (stay spend, not balance) |
| Skew fix | `np.log1p` on R, F, M **before** scaling |
| Scale | \(x'=(x-\mu)/\sigma\) on the log frame; keep \(\mu,\sigma\) for new guest |
| Inertia | \(J=\sum_i\|x_i-c_{\ell_i}\|^2\) on the **scaled** matrix |
| Elbow / sil. | plot \(J(k)\); this extract likes \(k=4\) (ops capacity + elbow) |
| Name bins | groupby Cluster → **median** R/F/M in original units |
| PCA | a slide of the 3-D scaled space; PC1 ≈ value, PC2 ≈ recency |

**Order:** clean → RFM → log1p → scale → choose \(k\) → fit → profile originals → PCA last.

**Not a yield / pricing decision.** Clustering groups similar RFM rows. A human still names the bins and owns the treatment.


## Desired outcome

![flowchart](tourrfm_flowchart.png)

1. Load `data/tour_bookings.csv`. Drop missing `GuestID`. Keep `Quantity > 0` (and `UnitRate > 0` for a strict stayed-night book).
2. `TotalSum = Quantity * UnitRate`. Parse `StayDate` with `%d.%m.%Y %H:%M`. Cast `GuestID` to int.
3. `snapshot_date = max(StayDate) + 1 day`. Aggregate Recency / Frequency / Monetary per guest.
4. `log1p` the three columns, then z-score. Do not overwrite the original RFM table.
5. Elbow + silhouette for \(k=1\ldots10\). Fit K-Means at the chosen \(k\) (we use 4).
6. Write `rfm["Cluster"] = labels`. Profile **medians** in original units and name the bins.
7. PCA to 2-D is a picture, not the model. Alternates, more practice, then turn the simulation knobs.


## 0. Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
    from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
    from sklearn.decomposition import PCA
    from sklearn.metrics import silhouette_score
    HAS_SK = True
except ImportError:
    HAS_SK = False
    print("sklearn not found — TourRFM.py NumPy fallbacks will run the clustering cells.")

import TourRFM as br

plt.rcParams["figure.figsize"] = (8, 4.5)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 20)
print("sklearn", HAS_SK, "| TourRFM helpers ready")


## 1. Why RFM then K-Means in a bank

Core bookings are the wrong grain for a relationship segment. One guest can post 12 card purchases on one day — that is activity, but Frequency in this project counts **distinct BookingNo**, not SKUs. RFM collapses the book to one row per `GuestID`:

- **Recency** — days since last stayed night (relative to a snapshot).
- **Frequency** — distinct bookings, not product-line count.
- **Monetary** — sum of stay spend in the window (not end-of-day balance, not NII).

K-Means then partitions that 3-column table. It assumes roughly spherical blobs in Euclidean space, which is why we log-transform the commercial-whale tail and standardise before we call `.fit`.


## 2. Load and clean

This extract has ~22% missing `GuestID` (ATM / guest bookings without a guest) and reversals (`Quantity < 0`, `BookingNo` starting with `R`). Those rows cannot join a relationship segment.

**Task 2.1** Read `data/tour_bookings.csv`. Print shape, dtypes, missing counts, and the `Quantity` / `UnitRate` extremes.

**Task 2.2** Drop missing `GuestID`. Keep `Quantity > 0`. Optionally drop `UnitRate <= 0` (waived fees). Build `TotalSum`. Parse dates with `format="%d.%m.%Y %H:%M"`. Cast `GuestID` to int.

Sanity: no remaining negatives, no missing IDs, stay spend and unique-guest counts printed.


In [ ]:
# Task 2.1 — inspect the raw extract
# raw = pd.read_csv("data/tour_bookings.csv")



In [ ]:
# Task 2.2 — clean into `sales`
# sales = ...
# sales["TotalSum"] = ...
# sales["StayDate"] = pd.to_datetime(..., format="%d.%m.%Y %H:%M")



## 3. Look at the book before you cluster

**Task 3.1** Top regions by stay spend. What share is Costa Rica + Mexico?

**Task 3.2** Histogram of line-item `TotalSum` (clip the axis — a few commercial wires dominate). This is *not* the RFM Monetary column yet.


In [ ]:
# Task 3.1 / 3.2 — region flow + a quick booking histogram



## 4. Build the RFM table

`snapshot_date` is one day after the last booking so Recency is a positive integer (the most recent guest gets 1, never 0).

**Task 4** Group by `GuestID`:

- Recency = `(snapshot - last StayDate).days`
- Frequency = `nunique(BookingNo)`
- Monetary = `sum(TotalSum)`

Store a DataFrame named `rfm` with columns `GuestID, Recency, Frequency, Monetary`. Print `describe()` and the raw skew.


In [ ]:
# Task 4 — snapshot + named aggregation → rfm



## 5. Log1p then scale

Monetary skew on this extract is ~16. Frequency is milder but still right-tailed. `np.log1p` compresses the commercial-whale tail; a z-score then puts days, bookings and dollars on the same footing.

**Task 5** Build `rfm_log` from the three columns via `np.log1p`. Fit a scaler and store `rfm_scaled`. Print log-skew and the scaled column means / stds (should be ~0 / ~1). Keep original `rfm` untouched.


In [ ]:
# Task 5 — rfm_log and rfm_scaled
# rfm_log, rfm_scaled, mu, sd = tr.log_scale(rfm)



## 6. Elbow and silhouette

Inertia always falls with k. The useful question is where the drop flattens, cross-checked with silhouette and with how many treatments Retail can fund.

**Task 6** For k = 1…10 store inertia. From k=2 also store silhouette. Plot both. Mark a candidate k.


In [ ]:
# Task 6 — elbow + silhouette lists and a two-axis plot
# ks, inertias = tr.elbow_inertias(rfm_scaled)



## 7. Fit K-Means and attach labels

The elbow on this extract bends hard at **k = 4** (silhouette also peaks there). Four bins match a realistic treatment budget: protect the core, nurture new guest, win-back the mid-book, leave or cheap-test the dormant.

**Task 7** Fit K-Means with k=4, seed 42, n_init=10 on `rfm_scaled`. Write the labels into **`rfm["Cluster"]`**.


In [ ]:
# Task 7 — k_optimal = 4; rfm["Cluster"] = ...



## 8. Profile in original units and name the bins

A centroid lives in log-scaled space. The email you send a region manager does not. Use **medians** — Monetary is still heavy-tailed in dollars.

**Task 8.1** `groupby("Cluster")` mean / median / count for Recency, Frequency, Monetary.

**Task 8.2** guest share vs spend share by cluster. Which bin is the loyal / repeat group?

**Task 8.3** Give each integer a business name (Loyal / repeat guests / New-light / Cooling repeaters / Lost). Write `rfm["Segment"]`.


In [ ]:
# Task 8 — cluster_summary, value concentration, Segment names



## 9. PCA is a slide, not the model

Three scaled columns → two principal components. Colour by Cluster.

**Task 9** PCA to 2-D on `rfm_scaled`. Build a frame with `PC1`, `PC2`, `Cluster`. Scatter. Print explained-variance ratios and the 3×2 loading matrix (rows = R, F, M).


In [ ]:
# Task 9 — PCA scatter + loadings
# Z, ev, comps = tr.pca2(rfm_scaled)



## 10. Alternate code that reaches the same idea

A. `fit_predict` instead of `fit` + `.labels_` (or `tr.kmeans_fit`).
B. Scale with `RobustScaler` after the same `log1p`. Does k=4 membership move?
C. Quintile RFM scores (`tr.rfm_quintile_scores`, Recency inverted).
D. `AgglomerativeClustering(n_clusters=4)` if sklearn is present.
E. NumPy SVD instead of `sklearn.decomposition.PCA` (`tr.pca2` already falls back).


In [ ]:
# Task 10 — pick at least two alternates and compare cluster sizes / medians



## 11. More practice

**P1.** Rebuild RFM on Costa Rica only. Refit k=4. Do the same four names still fit?

**P2.** Restrict bookings to the last 180 days before the snapshot. Recency is compressed; Frequency and Monetary shrink. What happens to the core share of flow?

**P3.** Score one new guest who is not in the table: Recency=7, Frequency=10, Monetary=2500. Transform with the **training** log1p + μ,σ, then predict.

**P4.** Drop Frequency and cluster on Recency + Monetary only. Which segment collapses?


In [ ]:
# Task 11 — P1 Costa Rica / P2 180-day window / P3 one new guest / P4 drop Frequency



## 12. Simulation — turn the knobs

Modify a few values and watch inertia / silhouette / core flow-share move.

Knobs:

- `k` — 2 to 8
- `n` — subsample of guest (500, 1500, all)
- `noise` — Gaussian σ added to `rfm_scaled`
- `n_init` — 1 vs 10
- `tenure_days` — rebuild Monetary/Frequency on a trailing window

Use `tr.simulate`. Plot inertia and silhouette against k at two noise levels.


In [ ]:
# Task 12 — knobs
K = 4
N = None
NOISE = 0.0
N_INIT = 10
SEED = 42

# print(tr.simulate(rfm_scaled, k=K, n=N, noise=NOISE, n_init=N_INIT, seed=SEED))



## Audience rewrite (Jočys checklist + McMurrey types)

| Audience | What they need | One sentence they should hear |
|----------|----------------|-------------------------------|
| Expert (revenue / CRM scientist) | inertia, silhouette, loadings, log+scale order | k=4 matches the elbow; PC1 (~80%) is value, PC2 (~18%) is recency; name bins from medians. |
| Technician (CRM / loyalty / PMS ops) | a four-row treatment table and the guest keys | Loyal / repeat guests = cluster 0 on this seed; suppress expensive win-back to cluster 2 until a human reviews. |
| Executive (GM / CMO / DMO director) | concentration, not eigenvalues | 16% of guests produce 82% of observed stay spend; cooling repeaters are the cheapest win-back. |
| Nonspecialist | a front-desk analogy | We sorted customers by *how lately, how often, how much they spent* — not by passport country or first name. |

Literacy: bars and a 2-D scatter, not a 3-D cube. Subject knowledge: do not expand RFM into PD. Time span: one slide of concentration + one treatment table.


## What this model can and cannot do

**Can**
- Collapse bookings to one RFM row per guest and group similar rows.
- Show flow concentration (loyal share of stay spend vs headcount).
- Give campaign ops four named bins and a reproducible seed.

**Cannot**
- Predict default, attrition, or next-product take-up.
- Replace a credit limit, a BSA typology, or a Reg-B / fair-lending review.
- Travel unchanged to another charter, currency, or year without refitting the scaler.
- Treat `Cluster == 1` as a permanent VIP-tier badge — ids shuffle if you change k or the seed.

**Top uses:** CRM design, retain-vs-nurture split, commercial-whale vs mass-retail separation, board concentration slides.
**Anti-uses:** automated fee waivers without a human, scoring a guest with a different scaler, clustering on raw unlogged Monetary, using the bin as a rack rate.


## Next steps

- Add product mix (DDA / card / wire share) as extra scaled columns.
- Try k=3 if Retail will only fund three treatments.
- Cap Monetary at the 99th percentile before log1p and re-check core size.
- Read `TourRFM_Strategy_Guide.docx` before you clone this onto another booking extract.
